# N31 — Strategy Orchestrator

End-to-end multi-agent supervisor. Integrates N25–N30 sub-agents through three
processing layers: dynamic MoE-style routing, Monte Carlo simulation over the
probabilistic outputs, and structured LLM synthesis of the final recommendation.

**Architecture**
- Layer 1: dynamic routing — decides which sub-agents to call each lap
- Layer 2: Monte Carlo simulation — samples from each sub-agent's uncertainty outputs
- Layer 3: LLM synthesis — aggregates all reasoning + MC scores into `StrategyRecommendation`

**References**: Heilmeier et al. (2020) ApplSci 10/4229 (MC motorsport sim),
Wang et al. (2024) MoA arXiv:2406.04692 (reasoning aggregation pattern),
Liu et al. (2024) DeLLMa arXiv:2402.02392 (decision under uncertainty with LLM).


---

## Step 0 — Setup & model loading

Imports, repo root, LLM client, FastF1 cache. `OrchestratorCFG` holds the
orchestrator's runtime constants: MC simulation parameters, MoE routing
thresholds, and the LLM client used in Layer 3 synthesis.

| Constant | Value | Purpose |
|---|---|---|
| `n_sim` | 500 | MC draws per strategy candidate |
| `sc_prob_threshold` | 0.30 | activates N30 if N27.sc_prob_3lap exceeds this |
| `risk_tolerance_default` | 0.5 | α in `score = α·E[S] + (1−α)·P10[S]` |


In [ ]:
import json, sys
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import fastf1
from pydantic import BaseModel, Field
from typing import Optional
from langchain_openai import ChatOpenAI
repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [9]:
@dataclass
class OrchestratorCFG:
    """Runtime configuration for the Strategy Orchestrator (N31).

    n_sim controls Monte Carlo draws per strategy candidate in Layer 2. 500 draws
    keep variance of the mean below 0.01 position units within lap-level latency.

    sc_prob_threshold is the N27.sc_prob_3lap cutoff above which N30 is activated
    to retrieve safety-car regulation context for the pit decision.

    risk_tolerance_default (α) weights expected value vs worst-case in the MC
    score: score(S) = α·E[S] + (1−α)·P10[S]. α=1.0 aggressive, α=0.0 conservative.

    temperature=0.0 ensures deterministic structured output from Layer 3 LLM.
    """

    model_name: str = "local-model"
    base_url: str = "http://localhost:1234/v1"
    temperature: float = 0.0
    n_sim: int = 500
    sc_prob_threshold: float = 0.30
    risk_tolerance_default: float = 0.5

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        self.llm = ChatOpenAI(
            model=self.model_name,
            base_url=self.base_url,
            api_key="lm-studio",
            temperature=self.temperature,
        )

In [10]:
CFG = OrchestratorCFG()

print(f"LLM          : {CFG.model_name} @ {CFG.base_url}")
print(f"N_SIM        : {CFG.n_sim}")
print(f"SC threshold : {CFG.sc_prob_threshold}")
print(f"α default    : {CFG.risk_tolerance_default}")
print(f"EXPORT_DIR   : {CFG.export_dir}")

LLM          : local-model @ http://localhost:1234/v1
N_SIM        : 500
SC threshold : 0.3
α default    : 0.5
EXPORT_DIR   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\agents


---

## Step 1 — `RaceState` + MoE routing

`RaceState` is the per-lap context object passed to the orchestrator. It is a
Pydantic `BaseModel` (not a dataclass) so that Pydantic validates field types and
ranges before any agent is called — bad inputs fail loudly at the boundary.

`_decide_agents_to_call` implements Layer 1 routing. N25, N26, N27 and N29 are
always activated. N28 and N30 are conditional:

| Condition | Agent activated |
|---|---|
| N26.warning_level == "PIT_SOON" | N28 |
| N29.alerts contains PROBLEM or WARNING radio intent | N28 |
| N27.sc_prob_3lap > CFG.sc_prob_threshold | N30 |
| N28 activated | N30 (regulation constraint check) |

Returns a `set[str]` of agent keys so the caller can branch without inspecting
the outputs again.


In [ ]:


class RaceState(BaseModel):
    """Per-lap context slice passed to the Strategy Orchestrator.

    driver identifies the driver whose strategy is being evaluated — all gap
    and pace features are relative to this driver.

    lap and total_laps enable race-percentage features used by N28 (lap_race_pct)
    and the MC simulation for fuel load estimation.

    compound and tyre_life are the current stint values forwarded to N26.

    gap_ahead_s and pace_delta_s are the primary inputs for N27 overtake scoring.

    weather fields (air_temp, track_temp, rainfall) are forwarded to N14 (SC model)
    as contextual features.

    radio_msgs and rcm_events are pre-filtered to the current lap ±1 window by the
    caller before passing to N29 — N31 does not do the filtering itself.

    risk_tolerance (α) is the MC score weight: score = α·E[S] + (1−α)·P10[S].
    Validated in [0, 1] by Pydantic; default 0.5 is neutral.
    """

    driver:         str
    lap:            int
    total_laps:     int
    position:       int
    compound:       str
    tyre_life:      int
    gap_ahead_s:    float
    pace_delta_s:   float
    air_temp:       float
    track_temp:     float
    rainfall:       bool = False
    radio_msgs:     list[dict] = Field(default_factory=list)
    rcm_events:     list[dict] = Field(default_factory=list)
    risk_tolerance: float = Field(default=0.5, ge=0.0, le=1.0)


def _decide_agents_to_call(
    tire_warning: str,
    sc_prob_3lap: float,
    radio_alerts: list[dict],
) -> set[str]:
    """Layer 1 MoE routing — returns set of conditional agent keys to activate.

    N25, N26, N27, N29 are always called by run_strategy_orchestrator and are
    not returned here. This function only decides N28 and N30.

    tire_warning is TireOutput.warning_level ("OK" | "MONITOR" | "PIT_SOON").
    sc_prob_3lap is RaceSituationOutput.sc_prob_3lap from N27.
    radio_alerts is RadioOutput.alerts — each dict has keys 'source' and 'intent'
    or 'event_type'.

    N28 is activated if the tyre is near the cliff or if radio signals a problem.
    N30 is activated whenever N28 is active (regulation check on the pit decision)
    or if SC probability is high (fetch SC procedure articles).
    """
    activate: set[str] = set()

    if tire_warning == "PIT_SOON":
        activate.add("N28")

    alert_intents = {a.get("intent", "") for a in radio_alerts}
    if alert_intents & {"PROBLEM", "WARNING"}:
        activate.add("N28")

    if sc_prob_3lap > CFG.sc_prob_threshold:
        activate.add("N30")

    if "N28" in activate:
        activate.add("N30")

    return activate


In [12]:
# Smoke test — routing logic
state = RaceState(
    driver="NOR", lap=18, total_laps=57, position=3,
    compound="C2", tyre_life=20, gap_ahead_s=1.2, pace_delta_s=-0.3,
    air_temp=32.0, track_temp=48.0, rainfall=False,
    radio_msgs=[], rcm_events=[], risk_tolerance=0.5,
)
print("RaceState OK:", state.driver, "lap", state.lap, "tyre_life", state.tyre_life)

# Case 1 — tyre cliff
a1 = _decide_agents_to_call("PIT_SOON", 0.10, [])
print("PIT_SOON, sc=0.10, no alerts  →", a1)  # {N28, N30}

# Case 2 — SC spike, no pit need
a2 = _decide_agents_to_call("MONITOR", 0.45, [])
print("MONITOR,  sc=0.45, no alerts  →", a2)  # {N30}

# Case 3 — radio PROBLEM
a3 = _decide_agents_to_call("OK", 0.10, [{"source": "radio", "intent": "PROBLEM"}])
print("OK,       sc=0.10, PROBLEM     →", a3)  # {N28, N30}

# Case 4 — all quiet
a4 = _decide_agents_to_call("MONITOR", 0.05, [])
print("MONITOR,  sc=0.05, no alerts  →", a4)  # set()


RaceState OK: NOR lap 18 tyre_life 20
PIT_SOON, sc=0.10, no alerts  → {'N28', 'N30'}
MONITOR,  sc=0.45, no alerts  → {'N30'}
OK,       sc=0.10, PROBLEM     → {'N28', 'N30'}
MONITOR,  sc=0.05, no alerts  → set()


### Step 1 results — `RaceState` + `_decide_agents_to_call`

`RaceState` instantiates and validates cleanly as a Pydantic `BaseModel`.

Routing smoke test — four cases:

| tire_warning | sc_prob_3lap | radio_alerts | agents activated |
|---|---|---|---|
| `PIT_SOON` | 0.10 | — | `{N28, N30}` |
| `MONITOR` | 0.45 | — | `{N30}` |
| `OK` | 0.10 | `PROBLEM` | `{N28, N30}` |
| `MONITOR` | 0.05 | — | `set()` |

N28 triggers whenever the tyre is near the cliff or a PROBLEM/WARNING alert
is present in the radio. N30 activates in cascade whenever N28 is active
(regulation constraint check before any pit recommendation) and independently
if sc_prob_3lap exceeds the 0.30 threshold. All four cases match the routing
rules in the architecture diagram.
